# Credit Card Launch Month Performance

## Problem Statement

Your team at a leading financial institution is preparing to launch a new credit card.

To estimate first-month issuance volume, analyze how previous credit cards performed during their launch month and return the card name along with the number of cards issued during that launch month.

## Input Table

### ccd_credit_cards

| Column Name | Data Type |
|------------|-----------|
| issue_month | INT |
| issue_year | INT |
| card_name | VARCHAR |
| issued_amount | INT |

## Requirements

- Analyze issuance history for each credit card.
- Identify the issuance amount corresponding to each card's launch month.
- Handle NULL values appropriately.
- Return results sorted by `issued_amount` in descending order.
- Return results matching the required output schema and order.

## Output Columns

| Column Name |
|------------|
| card_name |
| issued_amount |

## Sample Input

### ccd_credit_cards

| issue_month | issue_year | card_name | issued_amount |
|------------|------------|-----------|---------------|
| 1 | 2021 | Sapphire Plus | 170000 |
| 2 | 2021 | Sapphire Plus | 175000 |
| 3 | 2021 | Sapphire Plus | 180000 |
| 3 | 2021 | Freedom Flex | 65000 |

## Sample Output

| card_name | issued_amount |
|-----------|---------------|
| Sapphire Plus | 170000 |
| Freedom Flex | 65000 |

## Expected Output Schema

| Column Name | Data Type |
|------------|-----------|
| card_name | STRING |
| issued_amount | INT |

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql import Window

ccd_credit_cards_schema = StructType([
    StructField("issue_month", IntegerType(), True),
    StructField("issue_year", IntegerType(), True),
    StructField("card_name", StringType(), True),
    StructField("issued_amount", IntegerType(), True)
])

ccd_credit_cards_data = [
    (1, 2021, "Sapphire Plus", 170000),
    (2, 2021, "Sapphire Plus", 175000),
    (3, 2021, "Sapphire Plus", 180000),
    (3, 2021, "Freedom Flex", 65000)
]

ccd_credit_cards_df = spark.createDataFrame(
    ccd_credit_cards_data,
    schema=ccd_credit_cards_schema
)

In [0]:
result_df = (
    ccd_credit_cards_df.withColumn(
        "rnk",
        dense_rank().over(
            Window.partitionBy("card_name").orderBy(
                col("issue_year"), col("issue_month")
            )
        ),
    )
    .filter(col("rnk") == 1)
    .drop("rnk")
    .select("card_name","issued_amount")
    .orderBy(desc(col("issued_amount")))
)
display(result_df)